<!-- # Intro To Agents Part 1: Tools, MCPs & Tracing
--------------------

1. Introduction

2. Agents & Tools with LangChain

3. Model Context Protocols (MCPs) with FastMCP

4. Agents & Tracing with LangSmith

5. Next Steps -->

# Agents, Tools, MCPs and All That
---------

## 1. Introduction
-------------------

It's been a while since I posted. It's not as easy to write anymore with my family commitments, but its actually never been easier to do work with LLMs and AI more generally! Definitely the biggest developments in the last year or two have been on Agents and Agentic AI applications. To be honesy, I've always been a little skeptical about Agents and AI more generally. In science, they train you to be skeptical, as Feynman said, ["The first principle is that you must not fool yourself—and you are the easiest person to fool."](http://brainyquote.com/quotes/richard_p_feynman_137642) However, over the last two years I've very bought into AI and have switched roles to become an AI Engineer!

In this post I want to talk about Agents, Tools, MCPs and All That (the title being a play on the famous Vector Calculus book [Div, Grad, Curl and All that](https://www.google.com/books/edition/Div_Grad_Curl_and_All_that/sembQgAACAAJ?hl=en) that I read in undergrad) which are the newest crazes in AI and technology more broadly. I'll keep this post brief and simple. Partly because long posts are harder to write to, but also because people dont have attention anymore!

I'll go over how to buid a simple agent, use a [MCP](https://en.wikipedia.org/wiki/Model_Context_Protocol) server and observe agent behavoir; all using [LangChain](https://www.langchain.com/), [Groq](https://groq.com/), [FastMCP](https://gofastmcp.com/getting-started/welcome) and [LangSmith](https://www.langchain.com/langsmith-platform). The agent will be a simple ReAct agent. It will have tools that can help us find weather (like everyones first agent), but also help find the closest Police station and public restroom in NYC (data coming from [OpenData NYC](https://data.cityofnewyork.us/)). Very helpful things! In the back end, I'll use [MongoDB](https://www.mongodb.com/), [Redis](https://redis.io/) along with a handful of APIs to accompish these tasks.

Let's get into it, I'll first start with a bunch of import and then get into what an agent is:


In [1]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

/Users/mikeharmon/Desktop/agentmcps/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. Agents
-------------------
Large language models (LLMs) have been round for a few years at this point and most people are familiar with them through chatbots such as ChatGPT. With the advent of ChatGPT, people saw a AI model could have converations and write prose that was similar to a humans ability. LLMs were trained on an enormous amount of text data from the internet and can answer almost any question. I still remember in late 2022 asking ChatGPT to prove the uniqueness of solutions to the diffusion equation, it crushed the answer in 2 seconds. I asked more questiosn from semiconductor physics, quantum mechanics and numerical analysis, it got them all correct. I was impressed, but also realized it was reguritating things from the internet. Here's an example of a simple LLM with [LangChain](https://www.langchain.com/)

In [2]:
model = ChatGroq(model="openai/gpt-oss-120b")
llm  = model | StrOutputParser() 

print(llm.invoke("Give me a 100 word explanation of what a Sobolev Space is."))

A Sobolev space is a functional space that extends the concept of differentiability to functions whose derivatives may not be classically defined but exist in an integral sense. Formally, for an open set Ω⊂ℝⁿ and an integer k≥0, the Sobolev space W^{k,p}(Ω) consists of L^{p}(Ω) functions whose weak derivatives up to order k also belong to L^{p}(Ω). These spaces are Banach (Hilbert when p=2) and provide the natural setting for variational formulations of partial differential equations, embedding theorems, and regularity theory in practice. Sobolev spaces also enable trace theorems, allowing boundary values to be defined for functions lacking classical continuity.


Amazing right? 

Now ask it something simple:

In [3]:
print(llm.invoke("What happened on July 4th 2026?"))

I’m sorry, but I don’t have information about events that occurred on July 4, 2026. My training only includes data up through 2024, so I’m not able to provide details about that date. If you have a more specific question or need information about earlier years, I’d be happy to help.


While the model has been able to compress so much of humanity's knowledge into 120 billion parameters; it doesnt know something that happened after it was trained. An LLM deployed to answer questions for employees also doesnt know about things specific to your company.

Engineers solved this by using [Retrivial Augument Generation](https://michael-harmon.com/posts/rag_jfk2/), but this requires you to keep an up-to-date knowledge-base. People started introducing tools that allow the LLM to take actions (like search the web). Now the LLM can look up things it doesn't know, but it also allows it to take actions on your behalf and interact with its environment (like edit files, sumamrize emails, send messages, etc.). An agent is an LLM with the a set of tools. The simplest agent reasons about what steps to take based on a reques and its itnernal state. It can uses tools to take actions or generate text based on the internal state and the context (text and information you have provided as well as its collected). This is called a [ReAct agent](https://arxiv.org/abs/2210.03629).

The ability of a LLM to reason and also to take actions with tools has lead to a revolution in technology. We'll go over the basics of tools next.

## 3. Agents & Tools
-------------------
Let's go over first the tool everyone stars with, which is the ability to get the weather. A tool is a regular function; however it needs to be wrapped in a decorator to declare it one. We'll go over that in a second, but for now I'll import function:

In [ ]:
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)
from mymcp.server import get_weather

The defintion of this function is [here](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py). It is a simple function that takes in a city and queries the [OpenWeather API](https://openweathermap.org/) to give us the current weather:

In [22]:
get_weather("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 25.12,
  'feels_like': 25.2,
  'temp_min': 23.21,
  'temp_max': 26.19,
  'pressure': 1011,
  'humidity': 58,
  'sea_level': 1011,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 5.66, 'deg': 250},
 'clouds': {'all': 0},
 'dt': 1787532946,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1787480061,
  'sunset': 1787528603},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

How does an LLM call this function? That's where tools come in. They are decorators around the Python function that give the LLM enough information on when and how to use the function. The function annotations and the doc string are passed into the context window of the LLM so that it knows what its working with. 

We can create a weather tool for the LLM to use by importing from LangChain and then explictly make a tool called "`get_weather`" with the following (the decorator method with [FastMCP](http://gofastmcp.com/getting-started/welcome) is [here](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py)):

In [26]:
from langchain.tools import tool
weather_tool = tool("get_weather", get_weather)

The tool is actually now a co-routine that can be called with the async-invoke method:

In [27]:
await weather_tool.ainvoke("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 25.12,
  'feels_like': 25.2,
  'temp_min': 23.21,
  'temp_max': 26.19,
  'pressure': 1011,
  'humidity': 58,
  'sea_level': 1011,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 5.66, 'deg': 250},
 'clouds': {'all': 0},
 'dt': 1787532946,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1787480061,
  'sunset': 1787528603},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

Now we can use the [create_agent](https://reference.langchain.com/python/langchain/agents/factory/create_agent) to create a simple ReAct agent:

In [28]:
agents = create_agent(model=model, tools=[weather_tool])

Now I can pass the query in and use the agent using the same asynchronous `ainvoke` method,

In [29]:
query = "What is the weather in New York?"

In [30]:
result = await agents.ainvoke({'messages': [{'role': 'user', 'content': query}]})
messages = result.get("messages")


The ReAct agent returns a list of the all messages in the conversation:

In [31]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='9aa90351-2efe-4446-8350-f0b87e5d88c8'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the weather in New York?" Need to call get_weather function with city "New York". Use function.', 'tool_calls': [{'id': 'fc_1908a057-388e-4232-82d2-28c608f0632d', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 261, 'total_tokens': 317, 'completion_time': 0.118889854, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.022129046, 'prompt_tokens_details': None, 'queue_time': 0.078422765, 'total_time': 0.1410189}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0adba2bb92', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0316d-2483-75

Notice the conversation has a `HumanMessage` (prompt) -> `AIMessage` where the agent reasons what to do. In this case it only has one tool and calls it which returns a `ToolMessage`. The `ToolMessage` is read by the AI agent and synthesizes its message.  

With a long chain of reasoning and actions viewing the contents as a list isnt' easy. Instead most people use tracing and observability tools like [LangSmith](https://www.langchain.com/langsmith/observability), which I'll go over later.

## 4. Model Context Protocols (MCPs)
-------------------

A problem with tools is that they have to be localized and requires access to source code, data and other resources that developers might not wantt users to have access to. Additionally, each LLM and agent framework (OpenAI, Google, Anthropic, etc.) has their own way of defining and using tools. This can cause headaches for developers that need to make tools work across these various frameworks. The [**Model Context Protocol (MCP)**](https://en.wikipedia.org/wiki/Model_Context_Protocol) is an open standard for connecting LLM applications to external tools and data sources. Without MCP, each application must implement a custom integration for every tool. For example, a LangChain agent might need separate code for weather APIs, databases, file systems, and internal services.

MCP provides a common interface:

- **MCP server**: Exposes tools, resources, or prompts.
- **MCP client**: Connects an application or agent to the server.
- **Tool**: A callable operation, such as `get_weather`.
- **Schema**: Describes the tool's arguments and return values.

Model context protocol solves a few problems:

1. **Standardized integration**  
    Tools can be exposed through the same protocol instead of requiring custom integrations.

2. **Reusability**  
    One MCP server can be used by multiple agents and applications.

3. **Tool discovery**  
    Clients can list available tools and inspect their descriptions and input schemas.

4. **Separation of concerns**  
    The MCP server handles API calls and business logic, while the LLM application handles reasoning and conversation.

5. **Remote access**  
    Tools can run in another process or on another machine and be accessed over HTTP or other supported transports.

In some ways, MCP is for agents as the way APIs are for regular code. The standard MCP framework is [FastMCP](https://gofastmcp.com/getting-started/welcome). I created a FastMCP server here with several tools I'll discuss. 

First well create am MCP Client and connect to our server (which is running locally):

In [42]:
from fastmcp import Client
mcp_client = Client("http://localhost:8080/mcp")

Then we can get a list of the tools,

In [44]:
async with mcp_client:
    tools = await mcp_client.list_tools()

tools

[Tool(name='get_weather', title=None, description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', inputSchema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, outputSchema={'additionalProperties': True, 'type': 'object'}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='convert_address_to_point', title=None, description='Convert a street address to a Shapely Point object.', inputSchema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'description': 'A free‑form US address (e

These tools are of type [FastMCP tools](https://gofastmcp.com/servers/tools),

In [68]:
type(tools[0])

mcp.types.Tool

We can then call a specific tool using the client's [call_tool](https://gofastmcp.com/clients/tools#execution-options) method

In [56]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York"})

In [57]:
print(result)

CallToolResult(content=[TextContent(type='text', text='{"coord":{"lon":-74.006,"lat":40.7143},"weather":[{"id":803,"main":"Clouds","description":"broken clouds","icon":"04n"}],"base":"stations","main":{"temp":24.3,"feels_like":24.74,"temp_min":22.65,"temp_max":25.14,"pressure":1017,"humidity":75,"sea_level":1017,"grnd_level":1016},"visibility":10000,"wind":{"speed":4.47,"deg":163,"gust":5.36},"clouds":{"all":52},"dt":1787790329,"sys":{"type":1,"id":4610,"country":"US","sunrise":1787739438,"sunset":1787787529},"timezone":-14400,"id":5128581,"name":"New York","cod":200}', annotations=None, meta=None)], structured_content={'coord': {'lon': -74.006, 'lat': 40.7143}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 24.3, 'feels_like': 24.74, 'temp_min': 22.65, 'temp_max': 25.14, 'pressure': 1017, 'humidity': 75, 'sea_level': 1017, 'grnd_level': 1016}, 'visibility': 10000, 'wind': {'speed': 4.47, 'deg': 163, 'gust

Notice that it returns a [CallToolResult](https://gofastmcp.com/clients/tools#structured-results). This is all pretty much the standard example, so I'll show what is the closest public restroom to the point we received back:

In [58]:
lon = result.data.get("coord").get("lon")
lat = result.data.get("coord").get("lat")

async with mcp_client:
    restroom = await mcp_client.call_tool("find_closest_restroom", {"lat": lat, "lng": lon})

In [61]:
restroom.data

{'facility_name': 'New Amsterdam Library, NYPL',
 'location_type': 'Library',
 'operator': 'NYPL',
 'status': 'Operational',
 'open': 'Year Round',
 'hours_of_operation': 'Sunday: Closed \nMonday: 10:00 am - 7:00 pm \nTuesday: 10:00 am - 7:00 pm  \nWednesday: 10:00 am - 7:00 pm   \nThursday: 10:00 am - 7:00 pm   \nFriday: 10:00 am - 5:00 pm \nSaturday: 10:00 am - 5:00 pm',
 'accessibility': 'Fully Accessible',
 'restroom_type': 'Single-Stall All Gender Restroom(s)',
 'changing_stations': 'Yes',
 'latitude': None,
 'longitude': None,
 'website': 'https://www.nypl.org/locations/new-amsterdam'}

Great! Next let's give a LangChain agent access to these tools!

## 5. Agents & MCPs
-------------------

Now in order to make an LangChain agent have access to the tools supplied by an MCP server, then we need to use a specialized MCP client for LangChain called [LangChain MCP Adapter](https://reference.langchain.com/python/langchain-mcp-adapters). This can be imported and set to the address of the MCP server and set transport type:

In [63]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "http://localhost:8080/mcp", "transport": "http"}})

Now we can get the tools:

In [66]:
tool_list = await lc_client.get_tools()
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x12657fb00>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

Notice these are [LangChain tools](https://docs.langchain.com/oss/python/langchain/tools! The adapter converted the FastMCP tool to a LangChain one:

In [67]:
type(tool_list[0])

langchain_core.tools.structured.StructuredTool

We can then use the tool as before,

In [72]:
temperature = await tool_list[0].ainvoke({"city":"Boston"})

The immediate return value isnt quite the same, its a list json in it. We can convert back though,

In [77]:
import json
json.loads(temperature[0].get("text"))

{'coord': {'lon': -71.0598, 'lat': 42.3584},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04n'}],
 'base': 'stations',
 'main': {'temp': 22.4,
  'feels_like': 22.65,
  'temp_min': 20.64,
  'temp_max': 23.8,
  'pressure': 1017,
  'humidity': 75,
  'sea_level': 1017,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 2.57, 'deg': 140},
 'clouds': {'all': 86},
 'dt': 1787791539,
 'sys': {'type': 2,
  'id': 2007536,
  'country': 'US',
  'sunrise': 1787738592,
  'sunset': 1787786960},
 'timezone': -14400,
 'id': 4930956,
 'name': 'Boston',
 'cod': 200}

Now let's give the agent access to these tools from the MCP server. This just like giving the agent and other tools, 

In [78]:
agent = create_agent(model=model, tools=tool_list)

And now it can answer my question on temperature!

In [79]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'Whats the temparture in Boston?'
}]})
print(result.get("messages")[-1].content)

The current temperature in Boston is **about 22 °C** (≈ 72 °F).


Great let's try something less straight forward. The [tool to find the police precinct](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L89) reequires a latitude and longitude. If a users passes in an address the agent would need to conver the address to a latitude and longitude ([convert_address_to_point](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L57) tool) first and then pass that to the [get_police_precint tool](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L87). We can see this plainly with the entire message chain:

In [80]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattan, manhattan in?'
}]})

In [82]:
result.get("messages")

[HumanMessage(content='What police precinct is 306 West 54th Street, manhattan, manhattan in?', additional_kwargs={}, response_metadata={}, id='3fa87a68-b890-4b89-8a54-dfafc867ea54'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What police precinct is 306 West 54th Street, manhattan, manhattan in?" Need to determine precinct number. We have functions: convert_address_to_point, get_police_precinct. So first convert address to point, then get precinct. Use convert_address_to_point with address string. Then feed lat/lng to get_police_precinct. Then respond with precinct number and maybe info. Could also fetch precinct info using get_precinct_info for more detail. Let\'s do steps.\n\n', 'tool_calls': [{'id': 'fc_3d5d66e2-d976-410b-a5d2-ebc205032363', 'function': {'arguments': '{"address":"306 West 54th Street, Manhattan, NY"}', 'name': 'convert_address_to_point'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt

We can see that agent reasoning:

In [87]:
print(result.get("messages")[1].additional_kwargs.get("reasoning_content"))

The user asks: "What police precinct is 306 West 54th Street, manhattan, manhattan in?" Need to determine precinct number. We have functions: convert_address_to_point, get_police_precinct. So first convert address to point, then get precinct. Use convert_address_to_point with address string. Then feed lat/lng to get_police_precinct. Then respond with precinct number and maybe info. Could also fetch precinct info using get_precinct_info for more detail. Let's do steps.




Pretty cool! The react agent is reasoning well and uses the correct tool! 

One thing to note is that the MCP tools are all taking up space in our context window! This means the more tools we have, the more context we take up, which can be challenging. There are definitely strategies to mitigate this, but I wont go over this here. The final message is,


In [90]:
print(result.get("messages")[-1].content)

The address **306 West 54th Street, Manhattan, NY** falls within **NYPD Precinct 18**.


Looking at the agent actions as lists of message is not ideal. Instead we can use [LangSmith](https://www.langchain.com/langsmith-platform) to trace the agents reasons and tool usage.

<!--  -->

## 6. Agent Observability With Langsmith
-------------------

Parsing throught list of message is less than ideal. It also doesn't help us understand where and error occurred or what part of the process took what amount of time. LangSmith offers as a way to do all of this through there application. LangSmith integrates with most agent harnesses. It integreates seamlessly with LangChain and LangSmith.

As an example I can force the LLM to take 3 tool calls to get me the phone number of the closest police precinct (this requires the precinct number be sent to [get_precinct_info](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L207C5-L207C22) tool). The query is:

In [91]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What is the phone number for closests police station to 306 West 54th Street, Manhattan?'
}]})

The agent traces on LangSmith show us the path,

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/trace1.png?raw=1">

The agent clearly goes from `query` -> `convert_address_to_point` -> `get_police_precinct` -> `get_precinct_info`.

Zooming into the last tool call, we can see that the agent figured out how to pass the police precinct number into the tool:

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/trace2.png?raw=1">

It gets a json result back. The agent then generates a reponse that we see as,

In [92]:
print(result.get("messages")[-1].content)

The nearest NYPD precinct to 306 West 54th Street, Manhattan is the **Midtown North Precinct (Precinct 18)**.  

**Phone:** **212‑767‑8400**.


## 7. Next Steps
-------------------
In this quick post I went over how to create agents that use tools using Langchain and FastMCP and log the agent traces using LangSmith. This was all pretty simple and the MCP Server is local, so theres little value in using a MCP instead of pure tools. In a follow up post I'll show how to deploy this MCP server to Google Cloud and add it to Claude Code or another agent Harness.  

Hope you enjoyed this!